# Phase 1 Lab: Building the Compliance Data Collector

**Time Estimate:** 5-6 hours | **Prerequisites:** Phase 1 Theory notebook

**What you'll build:** Run the actual collector pipeline against mock data, verify the output, and understand how each piece connects.

**Important:** This lab imports from `src/` — the production source code. You're working with real, tested modules, not throwaway notebook code.

---

## Step 1: Setup & Verify Your Environment

First, make sure you can import the project modules. Run this from the project root directory.

In [3]:
import sys, os

# Add project root to path so we can import src/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"Source files exist: {os.path.exists(os.path.join(PROJECT_ROOT, 'src', 'models.py'))}")

Project root: c:\Users\jkl91\Documents\jonathanlohr-portfolio\jonathanlohr-portfolio\AWS Compliance collector
Source files exist: True


In [4]:
# Import the canonical data models (single source of truth)
from src.models import (
    EvidenceItem, CollectorResult, ScanResult, ControlAssessment,
    CompliancePosture, DriftEvent, ControlStatus,
    generate_scan_id, SEVERITY_WEIGHTS, CONTROL_FAMILIES
)

# Import the production modules we'll explore
from src.collector.security_hub import SecurityHubCollector
from src.collector.config_collector import ConfigCollector
from src.collector.iam_collector import IAMCollector
from src.collector.orchestrator import ComplianceCollector

print("All imports successful!")
print(f"NIST control families: {len(CONTROL_FAMILIES)}")
print(f"Severity levels: {list(SEVERITY_WEIGHTS.keys())}")

All imports successful!
NIST control families: 20
Severity levels: ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW', 'INFORMATIONAL']


## Step 2: Understand the Data Models

Before touching collectors, understand the data structures everything flows through.

**DDIA Connection (Ch. 4 — Encoding):** These dataclasses are our schema. Open `src/models.py` and read the docstrings.

In [5]:
# Create an EvidenceItem — the atomic unit of compliance evidence
evidence = EvidenceItem(
    source='security_hub',
    finding_id='arn:aws:securityhub:us-east-1:123456789012:finding-001',
    title='IAM.4 Root access key should not exist',
    status='FAILED',
    severity='CRITICAL',
    resource_type='AwsAccount',
    resource_id='123456789012',
    timestamp='2024-01-15T10:00:00Z',
    remediation='Delete root account access keys.',
    control_ids=['AC-2', 'AC-6(5)', 'IA-2(1)']  # Maps to 3 NIST controls
)

# Serialize and deserialize (what happens when we store to DynamoDB/S3)
serialized = evidence.to_dict()
print("Serialized evidence item:")
import json
print(json.dumps(serialized, indent=2))

# Round-trip test
restored = EvidenceItem.from_dict(serialized)
assert restored.control_ids == ['AC-2', 'AC-6(5)', 'IA-2(1)']
print(f"\nRound-trip OK: {restored.title}")

Serialized evidence item:
{
  "source": "security_hub",
  "finding_id": "arn:aws:securityhub:us-east-1:123456789012:finding-001",
  "title": "IAM.4 Root access key should not exist",
  "status": "FAILED",
  "severity": "CRITICAL",
  "resource_type": "AwsAccount",
  "resource_id": "123456789012",
  "timestamp": "2024-01-15T10:00:00Z",
  "remediation": "Delete root account access keys.",
  "control_ids": [
    "AC-2",
    "AC-6(5)",
    "IA-2(1)"
  ],
  "raw_data": {}
}

Round-trip OK: IAM.4 Root access key should not exist


In [6]:
# Scan IDs — sortable + unique
# System Design Interview (Ch. 7): Timestamp prefix = sortable, UUID = unique

ids = [generate_scan_id() for _ in range(5)]
import time
for scan_id in ids:
    print(scan_id)
    time.sleep(0.1)  # Small delay to see timestamp differences

print(f"\nNaturally sorted: {ids == sorted(ids)}")

2026-05-04T19-43-00Z_ec87502c
2026-05-04T19-43-00Z_da7d7608
2026-05-04T19-43-00Z_e3ad1587
2026-05-04T19-43-00Z_a8d8d0ba
2026-05-04T19-43-00Z_225b8f51

Naturally sorted: False


## Step 3: Explore the Security Hub Collector

Open `src/collector/security_hub.py` and read the code. Then let's test it.

In [7]:
# Read the actual source code and study it
with open(os.path.join(PROJECT_ROOT, 'src', 'collector', 'security_hub.py')) as f:
    source = f.read()

# Count key patterns
print("Code analysis:")
print(f"  Lines of code: {len(source.splitlines())}")
print(f"  Uses paginator: {'get_paginator' in source}")
print(f"  Handles throttling: {'ThrottlingException' in source}")
print(f"  Handles access denied: {'AccessDeniedException' in source}")
print(f"  Parses NIST refs: {'NIST.800-53' in source}")
print(f"  Retry config: {'adaptive' in source}")

# DVA-C02 Checkpoint: Can you explain WHY we use 'adaptive' retry mode
# instead of 'standard'? (Answer: adaptive adjusts retry timing based on
# the actual throttling response from AWS, giving better throughput)

Code analysis:
  Lines of code: 290
  Uses paginator: True
  Handles throttling: True
  Handles access denied: True
  Parses NIST refs: False
  Retry config: True


## Step 4: Explore the Config Collector

The Config collector has a critical piece of IP: the RULE_TO_NIST_MAP. This mapping is what you'd sell.

In [8]:
from src.collector.config_collector import ConfigCollector

# Examine the rule-to-NIST mapping
print(f"Total Config rules mapped: {len(ConfigCollector.RULE_TO_NIST_MAP)}")
print(f"\nRules by NIST family:")

family_counts = {}
for rule, controls in ConfigCollector.RULE_TO_NIST_MAP.items():
    for ctrl in controls:
        family = ctrl.split('-')[0]
        family_counts[family] = family_counts.get(family, 0) + 1

for family in sorted(family_counts.keys()):
    name = CONTROL_FAMILIES.get(family, 'Unknown')
    print(f"  {family} ({name}): {family_counts[family]} rule mappings")

print(f"\nExample mapping:")
for rule, controls in list(ConfigCollector.RULE_TO_NIST_MAP.items())[:5]:
    print(f"  {rule} → {controls}")

Total Config rules mapped: 32

Rules by NIST family:
  AC (Access Control): 7 rule mappings
  AU (Audit and Accountability): 5 rule mappings
  CM (Configuration Management): 3 rule mappings
  CP (Contingency Planning): 1 rule mappings
  IA (Identification and Authentication): 5 rule mappings
  SC (System and Communications Protection): 9 rule mappings
  SI (System and Information Integrity): 2 rule mappings

Example mapping:
  s3-bucket-public-read-prohibited → ['AC-3']
  s3-bucket-public-write-prohibited → ['AC-3']
  s3-bucket-server-side-encryption-enabled → ['SC-7']
  s3-bucket-versioning-enabled → ['SI-12']
  s3-object-lock-enabled → ['SC-7']


## Step 5: Explore the IAM Collector

Open `src/collector/iam_collector.py`. This collector provides evidence that Security Hub and Config can't — credential reports, password policy details, and account-level IAM statistics.

In [9]:
with open(os.path.join(PROJECT_ROOT, 'src', 'collector', 'iam_collector.py')) as f:
    iam_source = f.read()

print("IAM Collector analysis:")
print(f"  Lines of code: {len(iam_source.splitlines())}")
print(f"  Collects credential report: {'credential_report' in iam_source}")
print(f"  Collects password policy: {'password_policy' in iam_source}")
print(f"  Assesses NIST requirements: {'meets_nist' in iam_source or 'IA-5' in iam_source}")
print(f"\n  NIST controls covered: AC-2, IA-2, IA-4, IA-5(1)")
print(f"  DVA-C02 topics: IAM credential reports, password policies, MFA")

IAM Collector analysis:
  Lines of code: 488
  Collects credential report: True
  Collects password policy: True
  Assesses NIST requirements: True

  NIST controls covered: AC-2, IA-2, IA-4, IA-5(1)
  DVA-C02 topics: IAM credential reports, password policies, MFA


## Step 6: Run the Full Pipeline (Mock Data)

Now let's simulate what happens when the Lambda runs. We'll create realistic evidence, run it through the mapper, and generate a report.

**This is the end-to-end test the review identified as missing.**

In [10]:
from src.mapper.engine import ControlMappingEngine
from src.mapper.control_catalog import NIST_CONTROL_CATALOG
from src.evidence.pdf_generator import PDFReportGenerator
from src.drift.detector import DriftDetector
from datetime import datetime, timezone

# --- STEP 1: Create realistic evidence (simulates what collectors produce) ---

evidence_items = [
    EvidenceItem(
        source='security_hub', finding_id='sh-001',
        title='IAM.4 Root access key exists',
        status='FAILED', severity='CRITICAL',
        resource_type='AwsAccount', resource_id='123456789012',
        timestamp='2024-01-15T10:00:00Z',
        remediation='Delete root access keys.',
        control_ids=['AC-2', 'AC-6']
    ),
    EvidenceItem(
        source='config', finding_id='config-iam-user-mfa-enabled',
        title='Config: iam-user-mfa-enabled = NON_COMPLIANT',
        status='FAILED', severity='HIGH',
        resource_type='AWS::IAM::User', resource_id='user-without-mfa',
        timestamp='2024-01-15T10:00:00Z',
        remediation='Enable MFA for all IAM users.',
        control_ids=['AC-2', 'IA-2', 'IA-2(1)']
    ),
    EvidenceItem(
        source='config', finding_id='config-cloudtrail-enabled',
        title='Config: cloudtrail-enabled = COMPLIANT',
        status='PASSED', severity='INFORMATIONAL',
        resource_type='AWS::CloudTrail::Trail', resource_id='main-trail',
        timestamp='2024-01-15T10:00:00Z',
        control_ids=['AU-2', 'AU-3', 'AU-12']
    ),
    EvidenceItem(
        source='config', finding_id='config-encrypted-volumes',
        title='Config: encrypted-volumes = COMPLIANT',
        status='PASSED', severity='INFORMATIONAL',
        resource_type='AWS::EC2::Volume', resource_id='vol-abc123',
        timestamp='2024-01-15T10:00:00Z',
        control_ids=['SC-13', 'SC-28']
    ),
    EvidenceItem(
        source='security_hub', finding_id='sh-002',
        title='S3.5 SSL not required on bucket',
        status='FAILED', severity='HIGH',
        resource_type='AWS::S3::Bucket', resource_id='my-insecure-bucket',
        timestamp='2024-01-15T11:00:00Z',
        remediation='Add bucket policy requiring SSL.',
        control_ids=['SC-8']
    ),
    EvidenceItem(
        source='config', finding_id='config-guardduty-enabled',
        title='Config: guardduty-enabled = COMPLIANT',
        status='PASSED', severity='INFORMATIONAL',
        resource_type='AWS::GuardDuty::Detector', resource_id='detector-1',
        timestamp='2024-01-15T10:00:00Z',
        control_ids=['SI-4']
    ),
    EvidenceItem(
        source='config', finding_id='config-restricted-ssh',
        title='Config: restricted-ssh = COMPLIANT',
        status='PASSED', severity='INFORMATIONAL',
        resource_type='AWS::EC2::SecurityGroup', resource_id='sg-abc123',
        timestamp='2024-01-15T10:00:00Z',
        control_ids=['CM-6', 'SC-7']
    ),
    EvidenceItem(
        source='iam', finding_id='iam-password-policy',
        title='Password policy does NOT meet NIST requirements (8 chars, no rotation)',
        status='FAILED', severity='HIGH',
        resource_type='AWS::IAM::AccountPasswordPolicy', resource_id='password-policy',
        timestamp='2024-01-15T10:00:00Z',
        remediation='Set minimum 12 chars, require complexity, enforce 90-day rotation.',
        control_ids=['IA-5(1)']
    ),
]

print(f"Created {len(evidence_items)} evidence items")
print(f"Covering controls: {sorted(set(c for e in evidence_items for c in e.control_ids))}")

Created 8 evidence items
Covering controls: ['AC-2', 'AC-6', 'AU-12', 'AU-2', 'AU-3', 'CM-6', 'IA-2', 'IA-2(1)', 'IA-5(1)', 'SC-13', 'SC-28', 'SC-7', 'SC-8', 'SI-4']


In [11]:
# --- STEP 2: Build a ScanResult (what the orchestrator produces) ---

scan_result = ScanResult(
    scan_id=generate_scan_id(),
    scan_start=datetime.now(timezone.utc).isoformat(),
    account_id='123456789012',
    region='us-east-1'
)

# Add collector results
sh_items = [e for e in evidence_items if e.source == 'security_hub']
config_items = [e for e in evidence_items if e.source == 'config']
iam_items = [e for e in evidence_items if e.source == 'iam']

scan_result.collector_results['security_hub'] = CollectorResult(
    source='security_hub', status='SUCCESS',
    evidence_items=sh_items, raw_findings_count=len(sh_items)
)
scan_result.collector_results['config'] = CollectorResult(
    source='config', status='SUCCESS',
    evidence_items=config_items, raw_findings_count=len(config_items)
)
scan_result.collector_results['iam'] = CollectorResult(
    source='iam', status='SUCCESS',
    evidence_items=iam_items, raw_findings_count=len(iam_items)
)

# Finalize — aggregates all evidence, sets end timestamp and status
scan_result.finalize()

print(f"Scan ID: {scan_result.scan_id}")
print(f"Status: {scan_result.status}")
print(f"Total evidence items: {len(scan_result.all_evidence)}")
print(f"\nEvidence by control:")
for ctrl_id, items in sorted(scan_result.evidence_by_control.items()):
    statuses = [e.status for e in items]
    print(f"  {ctrl_id}: {len(items)} items — {statuses}")

Scan ID: 2026-05-04T19-46-33Z_76ee4bef
Status: COMPLETED
Total evidence items: 8

Evidence by control:
  AC-2: 2 items — ['FAILED', 'FAILED']
  AC-6: 1 items — ['FAILED']
  AU-12: 1 items — ['PASSED']
  AU-2: 1 items — ['PASSED']
  AU-3: 1 items — ['PASSED']
  CM-6: 1 items — ['PASSED']
  IA-2: 1 items — ['FAILED']
  IA-2(1): 1 items — ['FAILED']
  IA-5(1): 1 items — ['FAILED']
  SC-13: 1 items — ['PASSED']
  SC-28: 1 items — ['PASSED']
  SC-7: 1 items — ['PASSED']
  SC-8: 1 items — ['FAILED']
  SI-4: 1 items — ['PASSED']


In [ ]:
# --- STEP 3: Run the Mapping Engine ---

engine = ControlMappingEngine()
assessments = engine.assess_all_controls(scan_result)

print(f"Assessed {len(assessments)} controls")
print(f"\n{'Control':<12} {'Title':<40} {'Status':<15} {'Findings':<10} {'Priority'}")
print("-" * 90)
for a in sorted(assessments, key=lambda x: x.control_id):
    status_display = {
        'PASS': 'PASS', 'FAIL': '** FAIL **',
        'PARTIAL': '~ PARTIAL', 'NOT_ASSESSED': '? N/A', 'N/A': '- N/A'
    }.get(a.status.value, a.status.value)
    print(f"{a.control_id:<12} {a.control_title:<40} {status_display:<15} {a.total_findings:<10} {a.remediation_priority}")

TypeError: ControlMappingEngine() takes no arguments

In [ ]:
# --- STEP 4: Generate Compliance Posture ---

posture = engine.generate_posture(assessments)

print(f"COMPLIANCE POSTURE")
print(f"==================")
print(f"Score: {posture.compliance_percentage}%")
print(f"Total controls: {posture.total_controls}")
print(f"Applicable: {posture.applicable_controls}")
print(f"Passed: {posture.passed}  Failed: {posture.failed}  Partial: {posture.partial}  Not assessed: {posture.not_assessed}")
print(f"\nBy Family:")
for family, counts in sorted(posture.by_family.items()):
    print(f"  {family}: {counts}")
print(f"\nTop Failures (by priority):")
for f in posture.top_failures[:5]:
    print(f"  [{f['remediation_priority']}] {f['control_id']} — {f['control_title']}")

In [ ]:
# --- STEP 5: Generate Evidence PDF ---

pdf_gen = PDFReportGenerator()
output_path = os.path.join(PROJECT_ROOT, 'notebooks', 'output', 'compliance_report.txt')
result_path = pdf_gen.generate(assessments, posture, output_path)

print(f"Report generated: {result_path}")
print(f"File size: {os.path.getsize(result_path):,} bytes")
print(f"\nFirst 40 lines:")
print("=" * 60)
with open(result_path) as f:
    for i, line in enumerate(f):
        if i >= 40: break
        print(line.rstrip())

In [ ]:
# --- STEP 6: Simulate Drift Detection ---

# Create a "previous" scan where SC-8 was passing (now it's failing)
# and AU-2 was failing (now it's passing)
previous = []
for a in assessments:
    from copy import deepcopy
    prev = deepcopy(a)
    if a.control_id == 'SC-8':  # Was passing before
        prev.status = ControlStatus.PASS
        prev.failed_findings = 0
        prev.passed_findings = 1
    elif a.control_id == 'AU-2':  # Was failing before
        prev.status = ControlStatus.FAIL
        prev.failed_findings = 1
        prev.passed_findings = 0
    previous.append(prev)

detector = DriftDetector()
drift_events = detector.detect(previous, assessments)

print(f"Drift events detected: {len(drift_events)}")
print()
for event in drift_events:
    icon = {'REGRESSION': '!!', 'IMPROVEMENT': '++', 'NEW_FINDING': '??', 'RESOLVED': 'OK'}[event.drift_type]
    print(f"  [{icon}] {event.control_id}: {event.previous_status} → {event.current_status} ({event.drift_type})")
    print(f"       {event.details}")

## Step 7: Verify Everything with Tests

The project has 59 automated tests. Let's run them to confirm the pipeline is solid.

In [ ]:
!cd {PROJECT_ROOT} && python -m pytest tests/ -v --tb=short 2>&1 | tail -20

## Step 8: Your Exercises

### Exercise 1: Add a CloudTrail Collector
Create `src/collector/cloudtrail_collector.py` following the same pattern as `security_hub.py`. It should:
- Check if CloudTrail is enabled (`describe_trails`, `get_trail_status`)
- Return EvidenceItems mapped to AU-2, AU-3, AU-9, AU-12
- Handle the case where no trails exist

### Exercise 2: Add a New NIST Control
Add `CP-9` (System Backup) to `src/mapper/control_catalog.py`. You'll need:
- The control definition (look it up at https://csf.tools/reference/nist-sp-800-53/r5/)
- AWS evidence sources (hint: AWS Backup, RDS automated snapshots)
- Assessment criteria
Then add a Config rule mapping in `config_collector.py`

### Exercise 3: Write a Test
Add a test to `tests/test_mapper.py` that verifies your new CP-9 control is assessed correctly when you provide evidence for it.

### DVA-C02 Practice Questions
1. Our collector Lambda has a 15-minute timeout but sometimes takes 20 minutes for large accounts. What are two solutions? (Hint: Step Functions, or split into per-service Lambdas)
2. The `generate_credential_report()` API is async. How should you handle this in Lambda? (Poll with sleep, but watch remaining time via `context.get_remaining_time_in_millis()`)
3. Why do we use `ConditionExpression='attribute_not_exists(PK)'` when writing to DynamoDB? (Idempotency — prevents duplicate writes on Lambda retry)

---

## Next: Phase 2 — Control Mapping Engine Deep Dive

Open `phase-2-control-mapping/02-theory-and-lab-control-mapping.ipynb`

---

## Exercise 1 Simulation: Test the CloudTrail Collector

This section simulates the CloudTrail collector with mock data so you can verify it works without hitting real AWS. We use `moto` to fake the AWS API responses.

**Three scenarios tested:**
1. A fully configured trail (multi-region + logging on + validation) → should PASS
2. A trail with logging on but missing best practices → should PARTIAL
3. No trails at all → should FAIL

In [ ]:
from unittest.mock import MagicMock
from src.collector.cloudtrail_collector import CloudTrailCollector

# ── Scenario 1: Perfect trail (should PASS) ──────────────────────────────────
mock_client = MagicMock()

mock_client.describe_trails.return_value = {
    "trailList": [{
        "Name": "compliance-trail",
        "TrailARN": "arn:aws:cloudtrail:us-east-1:123456789012:trail/compliance-trail",
        "IsMultiRegionTrail": True,
        "LogFileValidationEnabled": True,
    }]
}
mock_client.get_trail_status.return_value = {"IsLogging": True}

collector = CloudTrailCollector(client=mock_client, account_id="123456789012")
result = collector.collect()

item = result.evidence_items[0]
print("── Scenario 1: Perfect trail ──")
print(f"  Status   : {item.status}")       # Expect: PASSED
print(f"  Severity : {item.severity}")     # Expect: INFORMATIONAL
print(f"  Controls : {item.control_ids}")  # Expect: AU-2, AU-3, AU-9, AU-12
print()

# ── Scenario 2: Logging on but single-region, no validation (should PARTIAL) ─
mock_client.describe_trails.return_value = {
    "trailList": [{
        "Name": "basic-trail",
        "TrailARN": "arn:aws:cloudtrail:us-east-1:123456789012:trail/basic-trail",
        "IsMultiRegionTrail": False,
        "LogFileValidationEnabled": False,
    }]
}
mock_client.get_trail_status.return_value = {"IsLogging": True}

collector2 = CloudTrailCollector(client=mock_client, account_id="123456789012")
result2 = collector2.collect()
item2 = result2.evidence_items[0]
print("── Scenario 2: Logging on, missing best practices ──")
print(f"  Status   : {item2.status}")    # Expect: PARTIAL
print(f"  Severity : {item2.severity}")  # Expect: MEDIUM
print()

# ── Scenario 3: No trails at all (should FAIL) ───────────────────────────────
mock_client.describe_trails.return_value = {"trailList": []}

collector3 = CloudTrailCollector(client=mock_client, account_id="123456789012")
result3 = collector3.collect()
item3 = result3.evidence_items[0]
print("── Scenario 3: No trails ──")
print(f"  Status   : {item3.status}")    # Expect: FAILED
print(f"  Severity : {item3.severity}")  # Expect: HIGH
print(f"  Title    : {item3.title}")
print()

print("All scenarios passed!" if all([
    item.status == "PASSED",
    item2.status == "PARTIAL",
    item3.status == "FAILED",
]) else "Something is wrong — check the output above.")

---

## Exercise 2 Simulation: CP-9 (System Backup) Control

CP-9 is the NIST control that says "you must have backups." We added it to three places:
1. **`control_catalog.py`** — defines what CP-9 is and what passing looks like
2. **`config_collector.py`** — maps 4 AWS Config backup rules to CP-9
3. **`test_mapper.py`** — verifies the control is assessed correctly

This simulation confirms all three pieces work together.

In [ ]:
from src.mapper.control_catalog import NIST_CONTROL_CATALOG
from src.collector.config_collector import ConfigCollector
from src.mapper.engine import ControlMappingEngine
from src.models import EvidenceItem, CollectorResult, ScanResult, ControlStatus, generate_scan_id
from datetime import datetime, timezone

# ── Part 1: Confirm CP-9 is in the catalog ───────────────────────────────────
print("── Part 1: Catalog check ──")
cp9 = NIST_CONTROL_CATALOG.get("CP-9")
assert cp9 is not None, "CP-9 missing from catalog!"
print(f"  Title      : {cp9['title']}")
print(f"  Family     : {cp9['family']}")
print(f"  Baselines  : {cp9['fedramp_baselines']}")
print(f"  Criteria   : {cp9['assessment_criteria'][:80]}...")
print()

# ── Part 2: Confirm Config rules are mapped to CP-9 ──────────────────────────
print("── Part 2: Config rule mapping check ──")
cp9_rules = [rule for rule, controls in ConfigCollector.RULE_TO_NIST_MAP.items()
             if "CP-9" in controls]
assert len(cp9_rules) > 0, "No Config rules mapped to CP-9!"
for rule in cp9_rules:
    print(f"  {rule} → CP-9")
print()

# ── Part 3: Simulate FAIL scenario (no backups) ───────────────────────────────
print("── Part 3: FAIL scenario (RDS backups disabled) ──")
scan_fail = ScanResult(
    scan_id=generate_scan_id(),
    scan_start=datetime.now(timezone.utc).isoformat(),
    account_id="123456789012",
    region="us-east-1",
)
scan_fail.collector_results["config"] = CollectorResult(
    source="config", status="SUCCESS",
    evidence_items=[
        EvidenceItem(
            source="config",
            finding_id="config-rds-backup-enabled",
            title="Config: rds-backup-enabled = NON_COMPLIANT",
            status="FAILED", severity="HIGH",
            resource_type="AWS::RDS::DBInstance",
            resource_id="prod-database",
            timestamp=datetime.now(timezone.utc).isoformat(),
            remediation="Enable automated backups with retention >= 7 days.",
            control_ids=["CP-9"],
        )
    ],
    raw_findings_count=1,
)
scan_fail.finalize()

engine = ControlMappingEngine()
assessments_fail = engine.assess_all_controls(scan_fail)
cp9_fail = next(a for a in assessments_fail if a.control_id == "CP-9")
print(f"  Status          : {cp9_fail.status.value}")   # Expect: FAIL
print(f"  Failed findings : {cp9_fail.failed_findings}") # Expect: 1
print(f"  Priority        : {cp9_fail.remediation_priority}")
print()

# ── Part 4: Simulate PASS scenario (backups configured) ──────────────────────
print("── Part 4: PASS scenario (backups properly configured) ──")
scan_pass = ScanResult(
    scan_id=generate_scan_id(),
    scan_start=datetime.now(timezone.utc).isoformat(),
    account_id="123456789012",
    region="us-east-1",
)
scan_pass.collector_results["config"] = CollectorResult(
    source="config", status="SUCCESS",
    evidence_items=[
        EvidenceItem(
            source="config",
            finding_id="config-rds-backup-enabled",
            title="Config: rds-backup-enabled = COMPLIANT",
            status="PASSED", severity="INFORMATIONAL",
            resource_type="AWS::RDS::DBInstance",
            resource_id="prod-database",
            timestamp=datetime.now(timezone.utc).isoformat(),
            control_ids=["CP-9"],
        )
    ],
    raw_findings_count=1,
)
scan_pass.finalize()

assessments_pass = engine.assess_all_controls(scan_pass)
cp9_pass = next(a for a in assessments_pass if a.control_id == "CP-9")
print(f"  Status          : {cp9_pass.status.value}")   # Expect: PASS
print(f"  Failed findings : {cp9_pass.failed_findings}") # Expect: 0
print()

print("Exercise 2 complete!" if all([
    cp9 is not None,
    len(cp9_rules) > 0,
    cp9_fail.status == ControlStatus.FAIL,
    cp9_pass.status == ControlStatus.PASS,
]) else "Something failed — check output above.")